In [ ]:

import streamlit as st
import pandas as pd
import plotly.express as px
from agent1_resume_parser import ResumeParserAgent
from agent2_question_generator import QuestionGeneratorAgent
from agent3_evaluator import AnswerEvaluatorAgent
import time

# Initialize session state
if 'initialized' not in st.session_state:
    st.session_state.initialized = True
    st.session_state.parser_agent = ResumeParserAgent()
    st.session_state.question_agent = QuestionGeneratorAgent()
    st.session_state.evaluator_agent = AnswerEvaluatorAgent()
    st.session_state.resume_text = ""
    st.session_state.technical_terms = []
    st.session_state.questions = []
    st.session_state.answers = []
    st.session_state.scores = []
    st.session_state.current_question_index = 0
    st.session_state.interview_complete = False

st.set_page_config(page_title="AI Technical Interview System", layout="wide")

# Custom CSS
st.markdown("""
<style>
.big-font { font-size:20px !important; }
.score-card { padding: 20px; border-radius: 10px; background-color: #f0f2f6; margin: 10px; }
.high-score { color: #28a745; }
.medium-score { color: #ffc107; }
.low-score { color: #dc3545; }
</style>
""", unsafe_allow_html=True)

st.title("🎯 AI Technical Interview System")
st.markdown("*Powered by 3 Specialized AI Agents*")

# Sidebar for resume upload
with st.sidebar:
    st.header("📄 Step 1: Upload Resume")
    uploaded_file = st.file_uploader("Choose PDF file", type="pdf")

    if uploaded_file and not st.session_state.questions:
        with st.spinner("Agent 1: Parsing resume and extracting technical terms..."):
            resume_text = st.session_state.parser_agent.extract_text_from_pdf(uploaded_file)
            terms = st.session_state.parser_agent.extract_technical_terms(resume_text)
            ranked_terms = st.session_state.parser_agent.rank_terms_by_relevance(terms, resume_text)
            st.session_state.technical_terms = ranked_terms

            st.success(f"✅ Extracted {len(ranked_terms)} technical terms")
            st.write("**Top Terms:**")
            for term in ranked_terms[:5]:
                st.write(f"- {term['term']} (Score: {term['relevance_score']})")

        with st.spinner("Agent 2: Generating interview questions..."):
            st.session_state.questions = st.session_state.question_agent.generate_questions(ranked_terms, max_questions=5)
            st.success(f"✅ Generated {len(st.session_state.questions)} questions")

# Main interview area
if st.session_state.questions and not st.session_state.interview_complete:
    current_q = st.session_state.current_question_index

    if current_q < len(st.session_state.questions):
        st.header(f"📋 Question {current_q + 1} of {len(st.session_state.questions)}")

        question_data = st.session_state.questions[current_q]

        col1, col2 = st.columns([3, 1])

        with col1:
            st.markdown(f"<div class='big-font'><b>Topic:</b> {question_data['term'].upper()}</div>", unsafe_allow_html=True)
            st.markdown(f"<div class='big-font'><b>Difficulty:</b> {question_data['difficulty']} 🔵</div>", unsafe_allow_html=True)
            st.info(f"❓ {question_data['question']}")

        with col2:
            st.metric("Expected Keywords", len(question_data['expected_keywords']))

        # Voice recording simulation
        if st.button("🎤 Record Answer", key=f"record_{current_q}"):
            with st.spinner("Agent 3: Listening and evaluating..."):
                answer = st.session_state.evaluator_agent.capture_voice_answer()
                evaluation = st.session_state.evaluator_agent.evaluate_answer(question_data, answer)

                st.session_state.answers.append(evaluation)
                st.session_state.scores.append(evaluation['total_score'])

                # Display evaluation
                st.subheader("📊 Evaluation Result")
                col_a, col_b, col_c = st.columns(3)
                with col_a:
                    st.metric("Total Score", f"{evaluation['total_score']}/100")
                with col_b:
                    st.metric("Keywords Found", f"{int(evaluation['keyword_score'] * 100)}%")
                with col_c:
                    st.metric("Confidence", f"{int(evaluation['confidence_score'] * 100)}%")

                st.write(f"**Feedback:** {evaluation['feedback']}")
                if evaluation['missing_keywords']:
                    st.warning(f"Missing keywords: {', '.join(evaluation['missing_keywords'])}")

                if st.button("Next Question →"):
                    st.session_state.current_question_index += 1
                    st.rerun()
    else:
        st.session_state.interview_complete = True
        st.rerun()

# Final report
if st.session_state.interview_complete or st.button("📊 Show Final Report"):
    if st.session_state.scores:
        st.balloons()
        st.header("— Interview Complete! Final Report")

        # Create results dataframe
        results_df = pd.DataFrame(st.session_state.answers)

        # Overall metrics
        col1, col2, col3, col4 = st.columns(4)
        with col1:
            st.metric("Overall Score", f"{sum(st.session_state.scores) / len(st.session_state.scores):.1f}/100")
        with col2:
            st.metric("Questions Answered", len(st.session_state.answers))
        with col3:
            st.metric("Avg Keyword Score", f"{results_df['keyword_score'].mean() * 100:.1f}%")
        with col4:
            st.metric("Strongest Topic", results_df.loc[results_df['total_score'].idxmax(), 'term'])

        # Score visualization
        fig = px.bar(results_df, x='term', y='total_score',
                     title='Question-wise Performance',
                     color='total_score', color_continuous_scale='Viridis',
                     labels={'total_score': 'Score', 'term': 'Technical Area'})
        st.plotly_chart(fig, use_container_width=True)

        # Detailed feedback
        st.subheader("— Detailed Analysis & Improvement Areas")

        weak_areas = results_df[results_df['total_score'] < 60]
        strong_areas = results_df[results_df['total_score'] >= 75]

        if not weak_areas.empty:
            st.error(" — **Areas Needing Improvement:**")
            for _, row in weak_areas.iterrows():
                st.write(f"- **{row['term']}**: {row['feedback']}")

        if not strong_areas.empty:
            st.success(" — **Strong Areas:**")
            for _, row in strong_areas.iterrows():
                st.write(f"- **{row['term']}**: Score {row['total_score']}/100")

        # Improvement suggestions
        st.subheader("— Personalized Learning Path")
        improvement_plan = []
        for _, row in results_df.iterrows():
            if row['total_score'] < 60:
                improvement_plan.append(f"— Review {row['term']} fundamentals and practice explaining {row['missing_keywords'][:2]}")

        if improvement_plan:
            for plan in improvement_plan[:3]:
                st.write(plan)
        else:
            st.write("— Excellent performance! Consider exploring advanced topics in your strong areas.")

        # Download report
        csv = results_df.to_csv(index=False).encode('utf-8')
        st.download_button("— Download Full Report (CSV)", csv, "interview_report.csv", "text/csv")

        if st.button("— Start New Interview"):
            for key in st.session_state.keys():
                del st.session_state[key]
            st.rerun()
    else:
        st.warning("No answers recorded yet. Please complete the interview first.")